# Разметка ЭКГ эксперимента 3

**Статус:** активный производитель кандидатных и принятых вручную R-зубцов для
записей РНЦХ. Автоматическая детекция не создаёт проверенную ЭКГ-разметку.

ЭКГ задаёт временную привязку сердечных циклов и не локализует источник
биоимпедансного сигнала.


## Входы и научный статус

Набор записей задаётся полем `annotation_targets` внешней конфигурации.
Для записей с дыхательным протоколом [`11.11`](11.11_Разметка_дыхания_эксперимента_3.ipynb)
должен предварительно создать и принять дыхательную разметку. Для технической
записи переключения каналов допускается ЭКГ-разметка без дыхательного
сопроводительного файла, если это явно указано конфигурацией.

Связь всех входов проверяется по полному SHA-256. Полярность ЭКГ выбирается
алгоритмом как диагностическое решение и затем проверяется человеком.

Расчётное окно электрической систолы строится по рабочей модели

$$
QT=QT_c\sqrt{RR},
$$

где $QT$ — расчётная продолжительность модельного окна; $RR$ — интервал между
соседними принятыми R-зубцами; $QT_c$ — заданный модельный параметр. Это окно
не является измеренной QT-разметкой, механической систолой или клапанным
событием.


In [ ]:
# Импорты и внешний контракт данных
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks

CONFIG_ENV = "KALMYKOV_EXP03_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp03_paths.example.json"
    )
CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
CSV_ROOT = (DATA_ROOT / CONFIG["csv_subdir"]).resolve()
CSV_ROOT.relative_to(DATA_ROOT)
BREATHING_DIR = DERIVED_ROOT / "exp03" / "annotations" / "breathing"
OUT_DIR = DERIVED_ROOT / "exp03" / "annotations" / "ecg"
OUT_DIR.mkdir(parents=True, exist_ok=True)

recording_items = CONFIG["recordings"]
record_ids = [item["record_id"] for item in recording_items]
if not recording_items or len(record_ids) != len(set(record_ids)):
    raise ValueError("recordings должен содержать уникальные record_id")
ECG_RECORDINGS = [
    item for item in recording_items if "ecg" in item.get("annotation_targets", [])
]

SOURCE_COLUMNS = CONFIG["source_columns"]
CANONICAL_COLUMNS = [
    "time_s", "rheo_1_mohm", "base_1_ohm", "qs_1_ohm",
    "ecg_v", "rheo_2_mohm", "base_2_ohm", "qs_2_ohm",
]
if len(SOURCE_COLUMNS) != len(CANONICAL_COLUMNS):
    raise ValueError("source_columns должен описывать восемь столбцов CSV")

PARAMETERS = {
    key: float(value) for key, value in CONFIG["ecg_detection"].items()
}
REQUIRED_PARAMETERS = {
    "low_hz", "high_hz", "filter_order", "min_rr_s", "peak_height_z",
    "peak_prominence_z", "refine_half_window_s", "qtc_s", "q_lead_s",
    "min_rr_for_qt_s", "default_rr_s",
}
if set(PARAMETERS) != REQUIRED_PARAMETERS:
    raise ValueError("ecg_detection должен содержать полный набор параметров")
for name, value in PARAMETERS.items():
    if not np.isfinite(value) or value <= 0:
        raise ValueError(f"Параметр {name} должен быть положительным")

ALGORITHM_VERSION = "exp03-ecg-rpeak-v2"


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def resolve_record_path(relative_path):
    path = (DATA_ROOT / relative_path).resolve()
    path.relative_to(CSV_ROOT)
    if not path.is_file():
        raise FileNotFoundError(f"Нет файла для записи: {relative_path}")
    return path


def read_record(path):
    frame = pd.read_csv(path)
    if list(frame.columns) != SOURCE_COLUMNS:
        raise ValueError("Схема CSV не совпадает с source_columns")
    frame.columns = CANONICAL_COLUMNS
    frame = frame.apply(pd.to_numeric, errors="raise")
    if len(frame) < 2 or not np.isfinite(frame.to_numpy(dtype=float)).all():
        raise ValueError("CSV пуст, слишком короток или содержит нечисловые значения")
    return frame


def sampling_frequency(frame):
    time = frame["time_s"].to_numpy(dtype=float)
    delta = np.diff(time)
    if len(delta) == 0 or np.any(~np.isfinite(delta)) or np.any(delta <= 0):
        raise ValueError("time_s должен быть конечным и строго возрастающим")
    median_dt = float(np.median(delta))
    jitter_fraction = float(np.median(np.abs(delta - median_dt)) / median_dt)
    return 1.0 / median_dt, jitter_fraction


In [ ]:
# Детекция R-зубцов и модельное окно электрической систолы
def detect_rpeaks(ecg_signal, fs_hz, parameters=PARAMETERS):
    low_hz = parameters["low_hz"]
    high_hz = parameters["high_hz"]
    nyquist_hz = fs_hz / 2.0
    if not 0 < low_hz < high_hz < nyquist_hz:
        raise ValueError("Полоса ЭКГ должна находиться ниже частоты Найквиста")
    order = int(parameters["filter_order"])
    if order < 1 or order != parameters["filter_order"]:
        raise ValueError("filter_order должен быть положительным целым")
    b, a = butter(
        order,
        [low_hz / nyquist_hz, high_hz / nyquist_hz],
        btype="band",
    )
    raw = np.asarray(ecg_signal, dtype=float)
    if len(raw) <= 3 * max(len(a), len(b)) or not np.isfinite(raw).all():
        raise ValueError("ЭКГ слишком короткая или содержит нечисловые значения")
    filtered = filtfilt(b, a, raw)
    scale = float(np.std(filtered))
    if not np.isfinite(scale) or scale <= np.finfo(float).eps:
        raise ValueError("После фильтрации отсутствует вариабельность ЭКГ")
    normalized = filtered / scale
    positive_strength = float(np.percentile(normalized, 99.5))
    negative_strength = float(np.percentile(-normalized, 99.5))
    polarity = 1 if positive_strength >= negative_strength else -1
    candidates, _ = find_peaks(
        polarity * normalized,
        distance=max(1, int(round(parameters["min_rr_s"] * fs_hz))),
        height=parameters["peak_height_z"],
        prominence=parameters["peak_prominence_z"],
    )
    half_window = max(1, int(round(parameters["refine_half_window_s"] * fs_hz)))
    signed_raw = polarity * raw
    refined = []
    for candidate in candidates:
        start = max(0, candidate - half_window)
        stop = min(len(raw), candidate + half_window + 1)
        refined.append(start + int(np.argmax(signed_raw[start:stop])))
    return np.asarray(sorted(set(refined)), dtype=int), int(polarity)


def model_electrical_systole(rpeaks_s, parameters=PARAMETERS):
    windows = []
    rpeaks_s = np.asarray(rpeaks_s, dtype=float)
    for index, r_time in enumerate(rpeaks_s):
        if index + 1 < len(rpeaks_s):
            rr_s = rpeaks_s[index + 1] - r_time
        elif index > 0:
            rr_s = r_time - rpeaks_s[index - 1]
        else:
            rr_s = parameters["default_rr_s"]
        qt_s = parameters["qtc_s"] * np.sqrt(
            max(rr_s, parameters["min_rr_for_qt_s"])
        )
        q_start_s = r_time - parameters["q_lead_s"]
        windows.append([float(q_start_s), float(q_start_s + qt_s)])
    return windows


In [ ]:
# Построение кандидатных ЭКГ-сопроводительных файлов
annotations = []
for spec in ECG_RECORDINGS:
    source_path = resolve_record_path(spec["relative_path"])
    input_sha256 = sha256_file(source_path)
    upstream_breathing = None
    if "breathing" in spec.get("annotation_targets", []):
        breathing_path = BREATHING_DIR / f"{spec['record_id']}.json"
        breathing = json.loads(breathing_path.read_text(encoding="utf-8"))
        if (
            breathing.get("record_id") != spec["record_id"]
            or breathing.get("algorithm_version") != "exp03-quiet-segments-v2"
            or breathing.get("input", {}).get("relative_path") != spec["relative_path"]
        ):
            raise RuntimeError(
                f"Неверное происхождение дыхательного sidecar: {spec['record_id']}"
            )
        if breathing.get("qc", {}).get("status") != "accepted":
            raise RuntimeError(f"Дыхательная разметка не принята: {spec['record_id']}")
        if not breathing.get("accepted_modes"):
            raise RuntimeError(f"Нет accepted_modes: {spec['record_id']}")
        if breathing.get("input", {}).get("sha256") != input_sha256:
            raise RuntimeError(f"Дыхательный sidecar относится к другому CSV")
        upstream_breathing = {
            "sidecar_sha256": sha256_file(breathing_path),
            "algorithm_version": breathing["algorithm_version"],
            "qc_status": breathing["qc"]["status"],
            "accepted_modes": breathing["accepted_modes"],
        }

    frame = read_record(source_path)
    fs_hz, jitter_fraction = sampling_frequency(frame)
    peak_indices, polarity = detect_rpeaks(
        frame["ecg_v"].to_numpy(dtype=float), fs_hz
    )
    time = frame["time_s"].to_numpy(dtype=float)
    candidate_rpeaks_s = [float(time[index]) for index in peak_indices]
    output = {
        "schema_version": 2,
        "annotation_type": "ecg",
        "algorithm_version": ALGORITHM_VERSION,
        "record_id": spec["record_id"],
        "subject_id": CONFIG["subject_id"],
        "role": spec["role"],
        "input": {
            "relative_path": spec["relative_path"],
            "sha256": input_sha256,
            "sampling_frequency_hz": fs_hz,
            "sampling_frequency_source": "median_diff_time_s",
            "relative_time_step_jitter": jitter_fraction,
        },
        "upstream_breathing": upstream_breathing,
        "candidate_rpeaks_s": candidate_rpeaks_s,
        "candidate_model_electrical_systole_s": model_electrical_systole(
            candidate_rpeaks_s
        ),
        "candidate_polarity": polarity,
        "accepted_polarity": None,
        "rpeaks_s": None,
        "model_electrical_systole_s": None,
        "model_parameters": PARAMETERS,
        "qc": {
            "status": "pending_manual_review",
            "reviewer": None,
            "reviewed_at": None,
            "notes": None,
            "history": [],
        },
    }
    output_path = OUT_DIR / f"{spec['record_id']}.json"
    if output_path.exists():
        existing = json.loads(output_path.read_text(encoding="utf-8"))
        if existing.get("input", {}).get("sha256") != input_sha256:
            raise RuntimeError(f"Конфликт входного SHA-256: {spec['record_id']}")
        if existing.get("algorithm_version") != ALGORITHM_VERSION:
            raise RuntimeError(
                f"ЭКГ-sidecar {spec['record_id']} создан другой версией алгоритма"
            )
        if existing.get("upstream_breathing") != upstream_breathing:
            raise RuntimeError(
                f"Изменился upstream breathing для {spec['record_id']}"
            )
        if (
            existing.get("qc", {}).get("status") == "accepted"
            and (
                not existing.get("rpeaks_s")
                or existing.get("accepted_polarity") not in {-1, 1}
            )
        ):
            raise RuntimeError(
                f"Принятый ЭКГ-sidecar {spec['record_id']} не содержит rpeaks_s или принятую полярность"
            )
        annotations.append(existing)
        print("Сохранён существующий ЭКГ-sidecar:", spec["record_id"], existing["qc"]["status"])
        continue
    output_path.write_text(
        json.dumps(output, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    annotations.append(output)
    print("Создан кандидат:", spec["record_id"], len(candidate_rpeaks_s))

if len(annotations) != len(ECG_RECORDINGS):
    raise RuntimeError("Создан не полный набор ЭКГ-sidecar-файлов")
print("ЭКГ-sidecar-файлов:", len(annotations))


## Ручной контроль качества и принятие разметки

Для каждой записи проверяются пропуски, ложные срабатывания, полярность и
участки с артефактами. Решение задаётся через `REVIEW_DECISION`. Статус
`accepted` требует имени проверяющего, явного списка `accepted_rpeaks_s` и
принятой полярности `accepted_polarity`.

После принятия модельные окна пересчитываются по проверенному списку. Статус
дыхательной разметки остаётся независимым входом и не принимает ЭКГ-разметку
автоматически. Повторный запуск не перезаписывает существующий
сопроводительный файл.


In [ ]:
# Ручной просмотр и явное принятие или отклонение ЭКГ-разметки
CHECK_RECORD_ID = None
REVIEW_DECISION = None
# Пример:
# REVIEW_DECISION = {
#     "record_id": "<record_id>",
#     "status": "accepted",
#     "reviewer": "<reviewer>",
#     "accepted_rpeaks_s": [1.02, 1.84, 2.67],
#     "accepted_polarity": 1,  # 1 или -1
#     "notes": "<основание решения>",
# }


def validate_rpeaks(rpeaks_s, start_s, stop_s):
    values = np.asarray(rpeaks_s, dtype=float)
    if values.ndim != 1 or len(values) < 2 or not np.isfinite(values).all():
        raise ValueError("Нужны не менее двух конечных R-зубцов")
    if np.any(np.diff(values) <= 0):
        raise ValueError("R-зубцы должны быть уникальны и строго упорядочены")
    if values[0] < start_s or values[-1] > stop_s:
        raise ValueError("R-зубцы выходят за границы записи")
    return [float(value) for value in values]


def apply_review(decision):
    record_id = decision["record_id"]
    sidecar_path = OUT_DIR / f"{record_id}.json"
    annotation = json.loads(sidecar_path.read_text(encoding="utf-8"))
    source_path = resolve_record_path(annotation["input"]["relative_path"])
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("SHA-256 исходного CSV изменился после разметки")
    status = decision["status"]
    reviewer = str(decision.get("reviewer", "")).strip()
    if status not in {"accepted", "rejected"} or not reviewer:
        raise ValueError("Нужны статус accepted/rejected и имя проверяющего")

    accepted_rpeaks_s = None
    accepted_polarity = None
    model_windows = None
    if status == "accepted":
        frame = read_record(source_path)
        time = frame["time_s"].to_numpy(dtype=float)
        accepted_rpeaks_s = validate_rpeaks(
            decision.get("accepted_rpeaks_s"),
            float(time[0]),
            float(time[-1]),
        )
        accepted_polarity = decision.get("accepted_polarity")
        if accepted_polarity not in {-1, 1}:
            raise ValueError("accepted_polarity должен быть равен 1 или -1")
        model_windows = model_electrical_systole(accepted_rpeaks_s)

    reviewed_at = datetime.now(timezone.utc).isoformat()
    previous_qc = annotation.get("qc", {})
    history = list(previous_qc.get("history", []))
    history.append({
        "status": previous_qc.get("status"),
        "reviewer": previous_qc.get("reviewer"),
        "reviewed_at": previous_qc.get("reviewed_at"),
        "notes": previous_qc.get("notes"),
    })
    annotation["rpeaks_s"] = accepted_rpeaks_s
    annotation["accepted_polarity"] = accepted_polarity
    annotation["model_electrical_systole_s"] = model_windows
    annotation["qc"] = {
        "status": status,
        "reviewer": reviewer,
        "reviewed_at": reviewed_at,
        "notes": decision.get("notes"),
        "history": history,
    }
    sidecar_path.write_text(
        json.dumps(annotation, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return annotation


if REVIEW_DECISION is not None:
    reviewed = apply_review(REVIEW_DECISION)
    print(reviewed["record_id"], reviewed["qc"]["status"])

if CHECK_RECORD_ID is None:
    print("Задайте CHECK_RECORD_ID для просмотра конкретной записи.")
else:
    annotation = json.loads(
        (OUT_DIR / f"{CHECK_RECORD_ID}.json").read_text(encoding="utf-8")
    )
    source_path = resolve_record_path(annotation["input"]["relative_path"])
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("SHA-256 исходного CSV изменился")
    frame = read_record(source_path)
    time = frame["time_s"].to_numpy(dtype=float)
    rpeaks_s = (
        annotation.get("rpeaks_s")
        if annotation.get("qc", {}).get("status") == "accepted"
        else annotation["candidate_rpeaks_s"]
    )
    fig, axis = plt.subplots(figsize=(14, 4))
    axis.plot(time, frame["ecg_v"], color="0.35", linewidth=0.7)
    for r_time in rpeaks_s:
        axis.axvline(r_time, color="red", linewidth=0.6)
    axis.set_xlabel("Время, с")
    axis.set_ylabel("ЭКГ, В")
    axis.set_title(
        f"{CHECK_RECORD_ID}: ЭКГ-разметка; QC={annotation['qc']['status']}"
    )
    plt.tight_layout()
    plt.show()
